# Khmer TTS — Multi-Speaker Demo (person1 / person2) — Self-Contained

Everything except the **Fish Speech engine itself** (an external open-source project,
too large to inline) is defined directly in this notebook's cells — no dependency on
the `Pich09/voice-clone` GitHub repo, no `scripts/*.py`, no local package. Just upload
this one `.ipynb` and Run All.

Trains a **Stage 1 Khmer base model** (multi-speaker, from the DDD dataset) and then
lets you generate speech from Khmer text **as a chosen person** by name (`person1`,
`person2`, ...).

**How speaker selection works**: Fish Speech is a codec-LM that supports in-context
voice cloning — you don't need a separate model per person. At generation time you
give it a short *reference* clip (audio + its transcript) of the person you want, plus
the new text to say, and it continues speaking in that voice. So `PERSON_MAP` below
just maps a friendly name (`person1`) to a `speaker_id` already present in the DDD
dataset — no retraining needed to add a new person, only a new reference clip.

Runs on Kaggle, Colab, or local (GPU strongly recommended for training; CPU is fine
for just generating once you already have a trained checkpoint).

**Kaggle-specific setup (do this before Run All):**
- **Settings (right sidebar) → Accelerator → GPU** (T4 x2 or P100).
- **Settings → Internet → On** — required for `git clone`, `pip install`, and
  every Hugging Face download below; off by default on a new notebook.
- **Add-ons → Secrets → add `HF_TOKEN`** if the DDD dataset or base checkpoint
  ever need auth (harmless to add even if not required).
- Kaggle GPU sessions are capped at **9 hours**; use `START_ROUND` (CONFIG cell)
  to split a long run across multiple sessions if `STAGE1_ROUNDS` won't finish
  in one sitting.
- Working data lives under `/kaggle/temp` (large scratch space, wiped when the
  session ends) rather than `/kaggle/working` (persisted but capped at 20GB) —
  run the "back up the final checkpoint" cell near the end to copy just the
  trained model into `/kaggle/working` so it survives as this notebook's Output.

**Colab-specific setup (do this before Run All):**
- **Runtime → Change runtime type → GPU** (T4 is fine; a Pro plan's A100/L4 is faster).
- **🔑 Secrets panel (left sidebar) → add `HF_TOKEN`** if the DDD dataset or base
  checkpoint ever need auth (harmless to add even if not required; grant this
  notebook access to the secret when prompted).
- Free-tier Colab disconnects after **~12h**, or sooner if idle — if
  `STAGE1_ROUNDS` won't finish in one sitting, use `START_ROUND` (CONFIG cell)
  to resume across sessions.
- Working data lives under `/content` (fast local disk, but wiped on
  disconnect/reclaim) — run the "back up the final checkpoint" cell near the
  end to copy just the trained model to Google Drive so it survives.

## 0 · Configuration — edit this cell, then Run All

In [ ]:
# ============================= CONFIG =============================
WORKDIR = ""   # blank = auto-pick per environment (see cell below)

DATASET        = "DDD-Cambodia/khmer-speech-dataset"
BASE_CKPT_REPO = "fishaudio/openaudio-s1-mini"
LORA_PRESET    = "r_32_alpha_16_fast"  # high-rank: Stage 1 needs to learn a language

# --- Progressive multi-round training ---
# Train a few hundred steps on a small slice of DDD first, then bring in more
# data and train more steps, repeating. Each round's finished checkpoint becomes
# the NEXT round's starting point (chained via LoRA-merge), so later rounds
# build on what earlier rounds already learned instead of starting over.
# `cumulative_samples` is a running total (not "new samples this round") --
# HF caches already-downloaded shards locally, so re-requesting a larger
# cumulative cap is cheap, not a repeat full download.
STAGE1_ROUNDS = [
    {"cumulative_samples": 200,   "steps": 300},
    {"cumulative_samples": 1000,  "steps": 800},
    {"cumulative_samples": 5000,  "steps": 3000},
]
MAX_SHARD_FILES = 200  # hard cap on parquet shards downloaded, across all rounds

# Resume from a specific round instead of starting over -- e.g. if round 1
# already finished (its merged checkpoint exists under
# models/khmer_base/round1/merged) and you only need to retry round 2 onward
# after fixing an error, set START_ROUND = 2. Leave at 1 for a fresh run.
START_ROUND = 1

# Which two (or more) speakers to expose as friendly names. Leave blank and run
# the "list available speakers" cell (after training) first -- it prints the
# exact folder names to copy in here (DDD speaker_ids get sanitized, e.g.
# "f-adt1-0001" -> "f_adt1_0001").
PERSON_MAP = {
    "person1": "f_adt1_0001",   # e.g. "f_adt1_0001"
    "person2": "m_adt1_0007",   # e.g. "m_adt1_0007"
}
# ====================================================================

## 1 · Environment & GPU check

In [ ]:
import subprocess, sys, os

# Kaggle ships a `google.colab` compatibility shim (so Colab-style snippets
# also run on Kaggle), which means `import google.colab` SUCCEEDS on Kaggle
# too -- checking Colab first would misdetect every Kaggle session as Colab.
# Kaggle's own markers (the /kaggle mount, or its KAGGLE_* env vars) are
# authoritative, so check those first and only fall back to the google.colab
# import when neither is present.
IN_KAGGLE = os.path.exists('/kaggle') or 'KAGGLE_KERNEL_RUN_TYPE' in os.environ
if IN_KAGGLE:
    IN_COLAB = False
else:
    try:
        import google.colab  # noqa: F401
        IN_COLAB = True
    except ImportError:
        IN_COLAB = False
IN_LOCAL = not IN_KAGGLE and not IN_COLAB
ENV_NAME = 'Colab' if IN_COLAB else ('Kaggle' if IN_KAGGLE else 'Local')
print('Environment:', ENV_NAME)

if IN_KAGGLE:
    # Kaggle notebooks default to NO internet access -- must be turned on
    # per-session under Notebook Settings -> Internet (right sidebar), or
    # every git clone / pip install / huggingface_hub download below fails.
    # Check it here (fast) rather than letting a later cell fail confusingly.
    try:
        import urllib.request
        urllib.request.urlopen('https://huggingface.co', timeout=10)
    except Exception as e:
        raise RuntimeError(
            'No internet access detected on Kaggle. Fix: open the notebook '
            'Settings panel (right sidebar) -> turn "Internet" ON -> re-run. '
            f'(underlying error: {e})'
        )
    # Kaggle GPU sessions are capped at 9 hours per session (12h for TPU) --
    # if a training run needs longer than that, use START_ROUND (see CONFIG)
    # to split it across multiple sessions instead of one long run.
    print('Kaggle session GPU time limit: 9h -- plan STAGE1_ROUNDS/START_ROUND accordingly.')

import torch

# TPU accelerators set one of these env vars (Colab/Kaggle both use the
# TPU_NAME / XRT_TPU_CONFIG convention; older Colab TPU runtimes used
# COLAB_TPU_ADDR). No torch.cuda equivalent check exists for TPU -- it's a
# separate device family entirely (XLA, not CUDA).
HAS_TPU = bool(os.environ.get('TPU_NAME') or os.environ.get('XRT_TPU_CONFIG')
               or os.environ.get('COLAB_TPU_ADDR'))

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    ACCELERATOR = 'gpu'
elif HAS_TPU:
    ACCELERATOR = 'tpu'
    print(
        'TPU detected, no CUDA GPU. UNSUPPORTED / EXPERIMENTAL: Fish Speech '
        'hardcodes accelerator="gpu" (fish_speech/configs/base.yaml) and its '
        'transformer/codec code was written against CUDA-specific ops, not '
        'torch_xla. Overriding trainer.accelerator=tpu (see section 9) may get '
        'further than doing nothing, but is very likely to still fail inside '
        "Fish Speech's own model code (e.g. attention kernels with no XLA path), "
        'or to run correctly but far slower than GPU due to XLA recompiling on '
        'every new sequence-length shape in this variable-length dataset. '
        'Proceeding only because you asked to try it -- watch the training log '
        'in section 9 for the actual failure point if/when it breaks.'
    )
else:
    raise RuntimeError(
        'No GPU or TPU detected. Fish Speech training hardcodes accelerator="gpu" '
        '(fish_speech/configs/base.yaml) and will crash the moment training starts '
        'if neither is visible -- better to fail here than after section 9 has already '
        'spent time downloading/cleaning data. Fix: Runtime -> Change runtime type -> '
        'GPU (Colab) or Settings -> Accelerator -> GPU/TPU (Kaggle), then Runtime -> Restart '
        'and run all again. (CPU-only is fine later, just for the generation cells once '
        'you already have a trained checkpoint.)'
    )


def run_step(args, **kwargs):
    """Run a command with this interpreter (sys.executable) for python scripts,
    raising loudly with real stdout/stderr on failure instead of silently
    marching on into a confusing failure several cells later."""
    result = subprocess.run(args, capture_output=True, text=True, **kwargs)
    # Print BOTH streams even on success -- loguru (used throughout fish-speech's
    # tools) logs to stderr by default, so printing only stdout silently drops
    # every informational/progress message these scripts produce.
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if result.returncode != 0:
        raise RuntimeError(
            f"Command failed (exit {result.returncode}): {' '.join(map(str, args))}\n"
            f"--- stdout ---\n{result.stdout}\n--- stderr ---\n{result.stderr}"
        )
    return result


def run_step_streaming(args, label=None, tail_lines=200, **kwargs):
    """Like run_step, but for long-running commands (e.g. training) where
    buffering all output until the end would hide live progress. Streams
    stdout+stderr line-by-line as they're produced, while also keeping the
    last `tail_lines` so a failure is self-explanatory instead of just an
    exit code -- Popen with inherited fds can lose output in notebook
    environments, so we deliberately pipe and re-print every line ourselves."""
    import collections
    tail = collections.deque(maxlen=tail_lines)
    proc = subprocess.Popen(args, stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT, text=True, bufsize=1, **kwargs)
    for line in proc.stdout:
        print(line, end='')
        tail.append(line)
    proc.wait()
    if proc.returncode != 0:
        raise RuntimeError(
            f"{label or args[0]} failed (exit {proc.returncode}). Last {len(tail)} lines:\n"
            + ''.join(tail)
        )
    return proc.returncode

## 2 · Working directory + Fish Speech engine

The only external clone left is `fishaudio/fish-speech` itself — the actual TTS
engine/training code. Everything *around* it (data prep, text normalization,
tokenizer patch, inference glue) is defined inline below.

In [ ]:
import os

if not WORKDIR:
    if IN_KAGGLE:
        # /kaggle/working is persisted as the notebook's Output but hard-capped
        # at 20GB -- this pipeline's dataset copies + checkpoints + LoRA
        # checkpoints across 3 rounds blow past that easily. /kaggle/temp is
        # scratch space on the same machine with much more room, but is NOT
        # persisted after the session ends -- copy the final merged checkpoint
        # to /kaggle/working yourself at the end (see the backup cell below).
        WORKDIR = '/kaggle/temp/khmer-voice-clone'
    elif IN_COLAB:
        # Local ephemeral disk, NOT Drive: Drive is capped at 15GB on the free
        # tier (this dataset + checkpoint + training artifacts blow past that
        # fast) and its FUSE mount doesn't reliably support the file operations
        # Lightning's checkpoint saving does (atomic renames etc.), which can
        # crash training outright. Local /content is ephemeral (wiped on
        # disconnect) but much bigger and a real filesystem. If you want the
        # final trained checkpoint to survive a disconnect, copy
        # MERGED_CKPT to Drive yourself at the end (see the last training cell).
        WORKDIR = '/content/khmer-voice-clone'
    else:
        WORKDIR = os.getcwd()

os.makedirs(WORKDIR, exist_ok=True)
os.chdir(WORKDIR)
print('WORKDIR:', WORKDIR)

FISH_DIR = os.path.join(WORKDIR, 'vendor', 'fish-speech')
if not os.path.isdir(os.path.join(FISH_DIR, '.git')):
    run_step(['git', 'clone', 'https://github.com/fishaudio/fish-speech', FISH_DIR])
else:
    run_step(['git', '-C', FISH_DIR, 'pull'])

## 3 · Install dependencies

In [ ]:
import importlib.util, os, re, subprocess, sys

_torch_ok = importlib.util.find_spec('torch') is not None


def pip_install(args, quiet=True):
    """Install one or more packages. On failure, retry as individual
    installs with full (non-quiet) output, so the error names the exact
    package that broke instead of an opaque batch failure across 30+ packages."""
    cmd = [sys.executable, '-m', 'pip', 'install'] + (['-q'] if quiet else []) + args
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode == 0:
        return
    if len(args) == 1:
        raise RuntimeError(f'pip install {args} failed:\n{result.stdout}\n{result.stderr}')
    print(f'Batch install failed, retrying {len(args)} packages one at a time to find the culprit...')
    for pkg in args:
        pip_install([pkg], quiet=False)


# Some packages we depend on (e.g. descript-audio-codec) still ship a legacy
# setup.py that calls `from distutils.core import setup`. Python 3.12 removed
# distutils from the standard library, so on newer Colab/Kaggle runtimes this
# fails with "python setup.py egg_info did not run successfully" unless
# setuptools' bundled distutils shim is used instead and pip/setuptools/wheel
# are recent enough to provide it.
pip_install(['--upgrade', 'pip', 'setuptools', 'wheel'])
os.environ['SETUPTOOLS_USE_DISTUTILS'] = 'local'

if not _torch_ok:
    pip_install(['torch', 'torchaudio'])

import torch
m = re.match(r'([\d.]+)(?:\+(\w+))?', torch.__version__)
torch_ver, cuda_tag = m.group(1), m.group(2)
ta_args = ['--force-reinstall', '--no-deps', f'torchaudio=={torch_ver}']
if cuda_tag:
    ta_args += ['--index-url', f'https://download.pytorch.org/whl/{cuda_tag}']
pip_install(ta_args)

pip_install(['--no-deps', '-e', FISH_DIR])
pip_install([
    'hydra-core', 'loguru', 'natsort', 'einops', 'rich', 'lightning', 'tensorboard',
    'loralib', 'pyrootutils', 'resampy', 'einx[torch]', 'zstandard', 'pydub',
    'ormsgpack', 'tiktoken', 'cachetools', 'safetensors', 'grpcio', 'kui',
    'opencc-python-reimplemented', 'modelscope', 'descript-audio-codec', 'gradio',
    'wandb', 'silero-vad', 'soundfile', 'numpy', 'huggingface_hub', 'datasets',
    'click', 'tqdm',
    'transformers==4.56.1', 'protobuf==4.25.5',
])

# Kaggle/Colab base images often ship hf_xet preinstalled, and some
# huggingface_hub versions still route large-file downloads through the Xet
# CDN even with HF_HUB_DISABLE_XET=1 set (a known inconsistency -- the env var
# alone doesn't reliably disable it once the package is present). Uninstalling
# hf_xet outright removes that code path entirely, forcing plain HTTP transfer
# for every hf_hub_download/snapshot_download call later in this notebook.
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'hf_xet'], capture_output=True, text=True)

print('Dependencies installed.')

## 4 · Base checkpoint (free, pretrained Fish Speech)

In [ ]:
import os

HF_TOKEN = ''
if IN_KAGGLE:
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
    except Exception as e:
        print('No HF_TOKEN in Kaggle Secrets (fine if public):', e)
elif IN_COLAB:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception as e:
        print('No HF_TOKEN in Colab Secrets (fine if public):', e)

# Fall back to a plain environment variable either way -- covers local runs,
# and lets you override/provide the token this way even on Kaggle/Colab
# (e.g. a secret named differently, or set via %env / os.environ in an
# earlier cell) without touching the platform-specific secrets store above.
if not HF_TOKEN:
    HF_TOKEN = os.environ.get('HF_TOKEN', '')
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
os.environ['HF_HUB_DISABLE_XET'] = '1'

from huggingface_hub import HfApi, snapshot_download
from huggingface_hub.utils import GatedRepoError, HfHubHTTPError
CHECKPOINT_DIR = os.path.join(WORKDIR, 'checkpoints', 'openaudio-s1-mini')
os.makedirs(CHECKPOINT_DIR, exist_ok=True)


def _find_corrupt_files(checkpoint_dir, repo_id, token):
    """A download can fail partway (this session hit a gated-repo 403 and a
    transient Xet CDN signature error on this exact repo) yet still leave a
    truncated/garbage file on disk that LOOKS present -- torch.load doesn't
    error on it, it just silently produces a near-empty state_dict, which only
    surfaces much later as "Missing key(s) in state_dict" for every single
    parameter once training tries to load it. Compare each file's size against
    what the Hub reports and flag any mismatch so it can be force-redownloaded
    instead of trusting "the file exists" as "the file is correct"."""
    api = HfApi(token=token)
    info = api.repo_info(repo_id, files_metadata=True)
    bad = []
    for f in info.siblings:
        if not f.size:
            continue
        local_path = os.path.join(checkpoint_dir, f.rfilename)
        actual_size = os.path.getsize(local_path) if os.path.isfile(local_path) else -1
        if actual_size != f.size:
            bad.append((f.rfilename, f.size, actual_size))
    return bad


# Kaggle's network occasionally drops mid-download on large files (the codec
# weights are ~1.9GB) -- snapshot_download itself has no retry around that, so
# one dropped connection kills the whole call. Retry a few times with backoff;
# local_dir downloads write a resumable .incomplete file, so a retry picks up
# from where it left off instead of restarting from zero. But a gated-repo/auth
# error will never succeed on retry (it's not transient), so fail immediately
# with clear instructions instead of burning ~2 minutes retrying it 5 times.
import time
_last_exc = None
for _attempt in range(5):
    try:
        snapshot_download(repo_id=BASE_CKPT_REPO, local_dir=CHECKPOINT_DIR,
                           token=os.environ.get('HF_TOKEN'), force_download=(_attempt > 0))
        bad_files = _find_corrupt_files(CHECKPOINT_DIR, BASE_CKPT_REPO, os.environ.get('HF_TOKEN'))
        if not bad_files:
            _last_exc = None
            break
        for rel_path, expected, actual in bad_files:
            print(f'  size mismatch: {rel_path} expected {expected} bytes, got {actual}')
        _last_exc = RuntimeError(f'{len(bad_files)} corrupt/incomplete file(s) after download')
    except (GatedRepoError, HfHubHTTPError) as e:
        if getattr(getattr(e, 'response', None), 'status_code', None) in (401, 403) or isinstance(e, GatedRepoError):
            raise RuntimeError(
                f'{BASE_CKPT_REPO} is a gated Hugging Face repo -- an unauthenticated '
                'or non-accepted token cannot download it. Fix:\n'
                f'  1. Log into huggingface.co and accept the license at '
                f'https://huggingface.co/{BASE_CKPT_REPO}\n'
                '  2. Create a read token at https://huggingface.co/settings/tokens\n'
                '  3. Kaggle: Add-ons -> Secrets -> add a secret named exactly HF_TOKEN with that value '
                '(Colab: Secrets panel, same name). Then re-run this cell.\n'
                f'Underlying error: {e}'
            ) from e
        _last_exc = e
    except Exception as e:
        _last_exc = e
    if _last_exc is not None:
        wait = 10 * (_attempt + 1)
        print(f'Download attempt {_attempt + 1}/5 failed ({_last_exc!r}), retrying in {wait}s (force_download next time)...')
        time.sleep(wait)
if _last_exc is not None:
    raise _last_exc
print('Base checkpoint at:', CHECKPOINT_DIR, '-- all files verified against Hub-reported sizes.')

## 5 · Patch a real upstream Fish Speech bug

`FishTokenizer.__init__` passes the raw `tokenizer.tiktoken` *file* to
`transformers.AutoTokenizer.from_pretrained()`, which has always required a
*directory* containing `tokenizer_config.json` -- `openaudio-s1-mini` never shipped
one, so this fails on every transformers version. Confirmed upstream bug, not a
version-pinning issue. This patches `FishTokenizer` to load the vocab directly via
`tiktoken` instead. Idempotent -- safe to re-run.

In [ ]:
import os

_tokenizer_path = os.path.join(FISH_DIR, 'fish_speech', 'tokenizer.py')
_MARKER = "# [khmer-voice-clone patch] tiktoken-file loader"

_PATCH_IMPORTS = f"""{_MARKER}
import os as _os


def _resolve_tiktoken_dir(model_path):
    \"\"\"FishTokenizer gets called two ways in fish-speech: with the tokenizer
    file itself (.../tokenizer.tiktoken) from the training config, and with
    the checkpoint *directory* from BaseTransformer.from_pretrained(). Handle
    both.\"\"\"
    if _os.path.isfile(model_path) and model_path.endswith(".tiktoken"):
        return _os.path.dirname(model_path)
    if _os.path.isdir(model_path) and _os.path.isfile(
        _os.path.join(model_path, "tokenizer.tiktoken")
    ):
        return model_path
    return None
"""

_PATCH_INIT = '''    def __init__(self, model_path: str):
        _tiktoken_dir = _resolve_tiktoken_dir(model_path)
        if _tiktoken_dir is not None:
            self._tokenizer = _TiktokenFileBackend(_tiktoken_dir)
        else:
            self._tokenizer = AutoTokenizer.from_pretrained(model_path)
        self.semantic_id_to_token_id = {}'''

_BACKEND_CLASS = f'''

{_MARKER}
class _TiktokenFileBackend:
    """Minimal AutoTokenizer-shaped wrapper around a raw tokenizer.tiktoken +
    special_tokens.json pair, for checkpoints that never shipped a
    tokenizer_config.json."""

    _PAT_STR = (
        r"""(?i:\x27s|\x27t|\x27re|\x27ve|\x27m|\x27ll|\x27d)|[^\\r\\n\\p{{L}}\\p{{N}}]?\\p{{L}}+|\\p{{N}}{{1,3}}"""
        r"""| ?[^\\s\\p{{L}}\\p{{N}}]+[\\r\\n]*|\\s*[\\r\\n]+|\\s+(?!\\S)|\\s+"""
    )

    def __init__(self, model_dir: str):
        import json as _json

        from tiktoken import Encoding as _Encoding
        from tiktoken.load import load_tiktoken_bpe as _load_tiktoken_bpe

        mergeable_ranks = _load_tiktoken_bpe(
            _os.path.join(model_dir, "tokenizer.tiktoken")
        )
        with open(
            _os.path.join(model_dir, "special_tokens.json"), encoding="utf-8"
        ) as f:
            special_tokens = _json.load(f)

        self._model_dir = model_dir
        self._special_tokens = special_tokens
        self._enc = _Encoding(
            name="fish-tiktoken-local",
            pat_str=self._PAT_STR,
            mergeable_ranks=mergeable_ranks,
            special_tokens=special_tokens,
        )
        self._vocab = {{
            tok.decode("utf-8", errors="replace"): idx
            for tok, idx in mergeable_ranks.items()
        }}
        self._vocab.update(special_tokens)

    def get_vocab(self):
        return dict(self._vocab)

    @property
    def vocab_size(self):
        return self._enc.n_vocab

    @property
    def pad_token_id(self):
        return self._special_tokens.get("<|pad|>")

    @property
    def eos_token_id(self):
        return self._special_tokens.get("<|end_of_text|>") or self._special_tokens.get(
            "<|endoftext|>"
        )

    def convert_tokens_to_ids(self, token):
        return self._vocab.get(token)

    def encode(self, text, add_special_tokens=False, allowed_special="all", **kwargs):
        return self._enc.encode(text, allowed_special=allowed_special)

    def decode(self, tokens, **kwargs):
        return self._enc.decode(tokens)

    def save_pretrained(self, path):
        import shutil

        _os.makedirs(path, exist_ok=True)
        shutil.copy(
            _os.path.join(self._model_dir, "tokenizer.tiktoken"),
            _os.path.join(path, "tokenizer.tiktoken"),
        )
        shutil.copy(
            _os.path.join(self._model_dir, "special_tokens.json"),
            _os.path.join(path, "special_tokens.json"),
        )
'''

with open(_tokenizer_path, encoding='utf-8') as f:
    _src = f.read()

if _MARKER in _src:
    print(f'{_tokenizer_path} already patched, skipping.')
else:
    _old_init = '''    def __init__(self, model_path: str):
        self._tokenizer = AutoTokenizer.from_pretrained(model_path)
        self.semantic_id_to_token_id = {}'''
    if _old_init not in _src:
        raise RuntimeError(
            'Could not find the expected FishTokenizer.__init__ body to patch -- '
            'vendor/fish-speech/fish_speech/tokenizer.py may have changed upstream.'
        )
    _src = _src.replace(
        'import torch\nfrom transformers import AutoTokenizer\n',
        'import torch\nfrom transformers import AutoTokenizer\n\n' + _PATCH_IMPORTS,
        1,
    )
    _src = _src.replace(_old_init, _PATCH_INIT, 1)
    _src = _src.rstrip('\n') + '\n' + _BACKEND_CLASS
    with open(_tokenizer_path, 'w', encoding='utf-8') as f:
        f.write(_src)
    print(f'Patched {_tokenizer_path}.')

## 6 · Khmer text normalization (inline)

Rewrites transcripts before training and before generation so the model only ever
sees spelled-out Khmer words, never raw digits/dates/currency symbols -- e.g.
`"$5.50"` -> spelled-out Khmer words for five dollars fifty cents.

In [ ]:
import re


# ---- numbers ----
KHMER_DIGITS = "០១២៣៤៥៦៧៨៩"
ARABIC_DIGITS = "0123456789"
_DIGIT_TRANS_KM_TO_AR = str.maketrans(KHMER_DIGITS, ARABIC_DIGITS)
_DIGIT_TRANS_AR_TO_KM = str.maketrans(ARABIC_DIGITS, KHMER_DIGITS)

_ONES = {
    0: "សូន្យ", 1: "មួយ", 2: "ពីរ", 3: "បី", 4: "បួន",
    5: "ប្រាំ", 6: "ប្រាំមួយ", 7: "ប្រាំពីរ", 8: "ប្រាំបី", 9: "ប្រាំបួន",
}
_TEENS_TENS = {
    10: "ដប់", 20: "ម្ភៃ", 30: "សាមសិប", 40: "សែសិប",
    50: "ហាសិប", 60: "ហុកសិប", 70: "ចិតសិប", 80: "ប៉ែតសិប", 90: "កៅសិប",
}
_SCALE = [
    (1_000_000_000, "ពាន់លាន"),
    (1_000_000, "លាន"),
    (1_000, "ពាន់"),
    (100, "រយ"),
]


def khmer_digits_to_arabic(text: str) -> str:
    return text.translate(_DIGIT_TRANS_KM_TO_AR)


def arabic_to_khmer_digits(text: str) -> str:
    return text.translate(_DIGIT_TRANS_AR_TO_KM)


def _two_digit_to_words(n: int) -> str:
    if n < 10:
        return _ONES[n]
    if n in _TEENS_TENS:
        return _TEENS_TENS[n]
    tens = (n // 10) * 10
    ones = n % 10
    return _TEENS_TENS[tens] + _ONES[ones]


def _three_digit_to_words(n: int) -> str:
    if n < 100:
        return _two_digit_to_words(n)
    hundreds = n // 100
    rest = n % 100
    out = _ONES[hundreds] + "រយ"
    if rest:
        out += _two_digit_to_words(rest)
    return out


def _int_to_words(n: int) -> str:
    if n == 0:
        return _ONES[0]
    if n < 0:
        return "ដក" + _int_to_words(-n)
    if n < 100:
        return _two_digit_to_words(n)
    if n < 1000:
        return _three_digit_to_words(n)
    parts = []
    remaining = n
    for scale_val, scale_word in _SCALE:
        if remaining >= scale_val:
            count = remaining // scale_val
            remaining = remaining % scale_val
            count_words = _int_to_words(count) if count > 1 else "មួយ" if count == 1 else ""
            if count_words:
                parts.append(count_words + scale_word)
    if remaining:
        parts.append(_int_to_words(remaining))
    return "".join(parts)


def number_to_khmer_words(value) -> str:
    if isinstance(value, str):
        value = khmer_digits_to_arabic(value.strip())
        value = float(value) if "." in value else int(value)
    if isinstance(value, float):
        is_negative = value < 0
        value = abs(value)
        int_part = int(value)
        frac_str = f"{value:.10f}".split(".")[1].rstrip("0")
        int_words = _int_to_words(int_part)
        if not frac_str:
            return ("ដក" if is_negative else "") + int_words
        frac_words = "".join(_ONES[int(d)] for d in frac_str)
        result = f"{int_words}ចុច{frac_words}"
        return ("ដក" + result) if is_negative else result
    return _int_to_words(int(value))


_NUMBER_RE = re.compile(r"(?<![\w.])-?(?:\d{1,3}(?:,\d{3})+|\d+)(?:\.\d+)?(?![\w])")
_KHMER_NUMBER_RE = re.compile(r"[០-៩]+(?:\.[០-៩]+)?")


def verbalize_numbers_in_text(text: str) -> str:
    def _replace_arabic(m):
        raw = m.group(0).replace(",", "")
        try:
            return number_to_khmer_words(raw)
        except ValueError:
            return m.group(0)

    def _replace_khmer(m):
        try:
            return number_to_khmer_words(m.group(0))
        except ValueError:
            return m.group(0)

    text = _NUMBER_RE.sub(_replace_arabic, text)
    text = _KHMER_NUMBER_RE.sub(_replace_khmer, text)
    return text


# ---- dates ----
_KHMER_MONTHS = {
    1: "មករា", 2: "កុម្ភៈ", 3: "មីនា", 4: "មេសា", 5: "ឧសភា", 6: "មិថុនា",
    7: "កក្កដា", 8: "សីហា", 9: "កញ្ញា", 10: "តុលា", 11: "វិច្ឆិកា", 12: "ធ្នូ",
}
_FULL_DATE_RE = re.compile(
    r"ថ្ងៃទី\s*([0-9០-៩]{1,2})\s*ខែ\s*([0-9០-៩]{1,2}|[ក-អ]+)\s*ឆ្នាំ\s*([0-9០-៩]{4})"
)
_NUMERIC_DATE_RE = re.compile(r"\b([0-3]?[0-9])[/\-]([01]?[0-9])[/\-]([12][0-9]{3})\b")
_YEAR_RE = re.compile(r"ឆ្នាំ\s*([0-9០-៩]{4})")


def _year_to_words(year: int) -> str:
    return number_to_khmer_words(year)


def _full_date_repl(m):
    day_raw, month_raw, year_raw = m.groups()
    day = int(khmer_digits_to_arabic(day_raw))
    year = int(khmer_digits_to_arabic(year_raw))
    if month_raw.isdigit() or all(c in "0123456789០១២៣៤៥៦៧៨៩" for c in month_raw):
        month_num = int(khmer_digits_to_arabic(month_raw))
        month_word = _KHMER_MONTHS.get(month_num, month_raw)
    else:
        month_word = month_raw
    return f"ថ្ងៃទី{number_to_khmer_words(day)} ខែ{month_word} ឆ្នាំ{_year_to_words(year)}"


def _numeric_date_repl(m):
    day, month, year = (int(x) for x in m.groups())
    month_word = _KHMER_MONTHS.get(month, str(month))
    return f"ថ្ងៃទី{number_to_khmer_words(day)} ខែ{month_word} ឆ្នាំ{_year_to_words(year)}"


def _year_only_repl(m):
    year = int(khmer_digits_to_arabic(m.group(1)))
    return f"ឆ្នាំ{_year_to_words(year)}"


def verbalize_dates_in_text(text: str) -> str:
    text = _FULL_DATE_RE.sub(_full_date_repl, text)
    text = _NUMERIC_DATE_RE.sub(_numeric_date_repl, text)
    text = _YEAR_RE.sub(_year_only_repl, text)
    return text


# ---- currency ----
_DOLLAR_RE = re.compile(r"\$\s*(-?(?:\d{1,3}(?:,\d{3})+|\d+)(?:\.\d{1,2})?)")
_RIEL_SYMBOL_SUFFIX_RE = re.compile(r"(-?(?:\d{1,3}(?:,\d{3})+|\d+)(?:\.\d+)?)\s*៛")
_RIEL_SYMBOL_PREFIX_RE = re.compile(r"៛\s*(-?(?:\d{1,3}(?:,\d{3})+|\d+)(?:\.\d+)?)")
_RIEL_WORD_RE = re.compile(r"(-?(?:\d{1,3}(?:,\d{3})+|\d+)(?:\.\d+)?)\s*រៀល")


def _dollar_to_words(m):
    raw = m.group(1).replace(",", "")
    if "." in raw:
        dollars_str, cents_str = raw.split(".")
        dollars = int(dollars_str)
        cents = int(cents_str.ljust(2, "0")[:2])
        dollar_words = number_to_khmer_words(dollars) + "ដុល្លារ"
        if cents:
            return f"{dollar_words} {number_to_khmer_words(cents)}សេន"
        return dollar_words
    return number_to_khmer_words(int(raw)) + "ដុល្លារ"


def _riel_to_words(m):
    raw = m.group(1).replace(",", "")
    value = float(raw) if "." in raw else int(raw)
    return number_to_khmer_words(value) + "រៀល"


def khmer_digits_to_arabic_preserving(text: str) -> str:
    def repl(m):
        return khmer_digits_to_arabic(m.group(0))
    text = re.sub(r"(?<=[\$៛])[០-៩,\.]+", repl, text)
    text = re.sub(r"[០-៩,\.]+(?=\s*៛)", repl, text)
    text = re.sub(r"[០-៩,\.]+(?=\s*រៀល)", repl, text)
    return text


def verbalize_currency_in_text(text: str) -> str:
    text = khmer_digits_to_arabic_preserving(text)
    text = _DOLLAR_RE.sub(_dollar_to_words, text)
    text = _RIEL_SYMBOL_PREFIX_RE.sub(_riel_to_words, text)
    text = _RIEL_SYMBOL_SUFFIX_RE.sub(_riel_to_words, text)
    text = _RIEL_WORD_RE.sub(_riel_to_words, text)
    return text


# ---- top-level normalizer ----
import unicodedata

ZERO_WIDTH_CHARS = "\u200b\u200c\u200d\ufeff"
_ZERO_WIDTH_RE = re.compile("[" + ZERO_WIDTH_CHARS + "]")
_WHITESPACE_RE = re.compile(r"[ \t\u00a0]+")
_MULTI_NEWLINE_RE = re.compile(r"\n{2,}")
_PERCENT_RE = re.compile(r"(-?\d+(?:\.\d+)?)\s*%")
_PUNCT_MAP = {
    "…": "។", "..": ".", "\u2018": "'", "\u2019": "'",
    "\u201c": '"', "\u201d": '"', "\u2013": "-", "\u2014": "-",
}
_DOUBLE_COENG_RE = re.compile("\u17d2\u17d2+")
_SENTENCE_SPLIT_RE = re.compile(r"(?<=[។!?])\s*")


def _fix_khmer_ordering(text: str) -> str:
    text = _DOUBLE_COENG_RE.sub("\u17d2", text)
    text = re.sub("\u17d2(?=\\s|$)", "", text)
    return text


def _clean_unicode_and_whitespace(text: str) -> str:
    text = unicodedata.normalize("NFC", text)
    text = _ZERO_WIDTH_RE.sub("", text)
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = _WHITESPACE_RE.sub(" ", text)
    text = _MULTI_NEWLINE_RE.sub("\n", text)
    return text.strip()


def _normalize_punctuation(text: str) -> str:
    for src, dst in _PUNCT_MAP.items():
        text = text.replace(src, dst)
    return text


def _verbalize_percent(text: str) -> str:
    def repl(m):
        num = m.group(1)
        value = float(num) if "." in num else int(num)
        return number_to_khmer_words(value) + "ភាគរយ"
    return _PERCENT_RE.sub(repl, text)


def normalize_khmer_text(text: str) -> str:
    """Full normalization pipeline. Idempotent. Run identically before
    training, validation, and inference."""
    if not text:
        return text
    text = _clean_unicode_and_whitespace(text)
    text = _fix_khmer_ordering(text)
    text = _normalize_punctuation(text)
    text = verbalize_currency_in_text(text)
    text = verbalize_dates_in_text(text)
    text = _verbalize_percent(text)
    text = verbalize_numbers_in_text(text)
    text = _WHITESPACE_RE.sub(" ", text).strip()
    return text


def split_sentences(text: str):
    return [s.strip() for s in _SENTENCE_SPLIT_RE.split(text) if s.strip()]


print(normalize_khmer_text("ខ្ញុំមាន $10 នៅឆ្នាំ 2026។ តម្លៃបញ្ចុះ 25% សម្រាប់អតិថិជនថ្មី!"))

## 7 · Data pipeline: download DDD -> QC -> clean -> Fish format (inline)

Each stage is a plain Python function writing/reading JSONL manifests under
`data/manifests/`, matching the same manifest schema used by the main repo
(`audio_path`, `text`, `speaker_id`, `duration`, ...), so you can inspect any
intermediate stage on disk.

In [ ]:
import io, json, os

MANIFEST_DIR = os.path.join(WORKDIR, 'data', 'manifests')
os.makedirs(MANIFEST_DIR, exist_ok=True)


def list_shard_files(dataset, split):
    """List+sort the split's parquet shards ourselves, rather than letting
    `datasets` resolve/interleave/prefetch across every shard-numbering
    generation a repo has ever had -- that caused far more downloads than
    max_samples should need."""
    from huggingface_hub import HfApi
    api = HfApi(token=os.environ.get('HF_TOKEN'))
    files = api.list_repo_files(dataset, repo_type='dataset')
    prefix = f'data/{split}-'
    return sorted(f for f in files if f.startswith(prefix) and f.endswith('.parquet'))


def _hf_hub_download_with_retry(repo_id, repo_type, filename, token, attempts=5):
    """hf_hub_download over HF's Xet CDN occasionally 403s with a signature
    error ("invalid key pair id") -- usually a transient failure on HF's
    storage backend (a stale/rotated signing key on their edge), not a
    permissions problem despite the error text, and not something disabling
    Xet client-side avoids since the resolved URL itself is Xet-hosted for
    these repos. Retrying after a short wait clears most occurrences.
    Occasionally it's NOT transient -- the same content-hash keeps failing
    across every retry with a fresh signed URL each time, meaning that
    specific stored blob has a broken signing config server-side. Callers
    should treat a raised exception here as "this file is currently
    unavailable" and skip it rather than assume one more retry will help."""
    import time
    from huggingface_hub import hf_hub_download
    last_exc = None
    for attempt in range(attempts):
        try:
            return hf_hub_download(repo_id=repo_id, repo_type=repo_type, filename=filename, token=token)
        except Exception as e:
            last_exc = e
            wait = 5 * (attempt + 1)
            print(f'  hf_hub_download({filename}) attempt {attempt + 1}/{attempts} failed ({e!r}), retrying in {wait}s...')
            time.sleep(wait)
    raise last_exc


def rows_from_shard(dataset, shard_file, split):
    from datasets import Audio, load_dataset
    local_path = _hf_hub_download_with_retry(repo_id=dataset, repo_type='dataset', filename=shard_file,
                                              token=os.environ.get('HF_TOKEN'))
    shard_ds = load_dataset('parquet', data_files={split: local_path}, split=split)
    if 'audio' in shard_ds.features:
        shard_ds = shard_ds.cast_column('audio', Audio(decode=False))
    yield from shard_ds


def download_ddd(dataset, split, out_dir, max_samples=None, max_shard_files=100):
    import soundfile as sf
    audio_dir = os.path.join(out_dir, 'audio')
    os.makedirs(audio_dir, exist_ok=True)
    manifest_path = os.path.join(MANIFEST_DIR, 'ddd_raw.jsonl')
    n_written = 0

    def export_row(i, row, manifest_f):
        nonlocal n_written
        audio = row.get('audio')
        text = row.get('transcript') or row.get('text') or ''
        speaker_id = row.get('speaker_id') or row.get('speaker') or 'unknown'
        if audio is None or not text.strip():
            return
        if 'array' in audio:
            array, sr = audio['array'], audio['sampling_rate']
        else:
            data = audio.get('bytes')
            source = io.BytesIO(data) if data else audio['path']
            array, sr = sf.read(source)
        fname = f'{speaker_id}_{i:07d}.wav'
        out_path = os.path.join(audio_dir, fname)
        sf.write(out_path, array, sr)
        record = {
            'audio_path': out_path, 'text': text.strip(), 'speaker_id': speaker_id,
            'duration': round(len(array) / sr, 3), 'source': dataset,
        }
        manifest_f.write(json.dumps(record, ensure_ascii=False) + '\n')
        n_written += 1
        if n_written % 500 == 0:
            print(f'  ... {n_written} samples exported')

    with open(manifest_path, 'w', encoding='utf-8') as manifest_f:
        if max_samples:
            print(f'Listing shard files for {dataset} [{split}] ...')
            shard_files = list_shard_files(dataset, split)
            if not shard_files:
                raise RuntimeError(f'No shard files found matching data/{split}-*.parquet in {dataset}')
            if max_shard_files:
                shard_files = shard_files[:max_shard_files]
            print(f'{len(shard_files)} shard file(s) available (cap: {max_shard_files or "none"}).')
            i = 0
            n_shards_skipped = 0
            for shard_file in shard_files:
                if n_written >= max_samples:
                    break
                print(f'  fetching {shard_file} ...')
                try:
                    for row in rows_from_shard(dataset, shard_file, split):
                        if n_written >= max_samples:
                            break
                        export_row(i, row, manifest_f)
                        i += 1
                except Exception as e:
                    # A single shard can be permanently unavailable (a broken
                    # signing config on one specific HF Xet CDN blob, seen in
                    # practice -- retries inside rows_from_shard already ruled
                    # out a transient blip). With `max_shard_files` shards to
                    # pick from and only `max_samples` needed, skipping this
                    # one and moving on is safe and keeps the round moving
                    # instead of aborting the whole pipeline over one file.
                    n_shards_skipped += 1
                    print(f'  SKIPPING {shard_file} -- unavailable after retries ({e!r})')
                    continue
            if n_shards_skipped:
                print(f'Skipped {n_shards_skipped}/{len(shard_files)} unavailable shard file(s).')
        else:
            from datasets import Audio, load_dataset
            print(f'Loading full {dataset} [{split}] (streaming) ...')
            ds = load_dataset(dataset, split=split, streaming=True)
            if 'audio' in ds.features:
                ds = ds.cast_column('audio', Audio(decode=False))
            for i, row in enumerate(ds):
                export_row(i, row, manifest_f)

    print(f'Done. Wrote {n_written} records to {manifest_path}')
    return manifest_path


def _read_jsonl(path):
    with open(path, encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]


def _write_jsonl(path, rows):
    os.makedirs(os.path.dirname(path) or '.', exist_ok=True)
    with open(path, 'w', encoding='utf-8') as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + '\n')


def export_audio(inputs, output):
    """Re-check audio exists and (re)compute duration directly from file,
    merging one or more raw manifests into one."""
    import soundfile as sf
    n_in, n_out, out_rows = 0, 0, []
    for path in inputs:
        for row in _read_jsonl(path):
            n_in += 1
            if not os.path.exists(row['audio_path']):
                continue
            try:
                info = sf.info(row['audio_path'])
                row['duration'] = round(info.frames / info.samplerate, 3)
                row['sample_rate'] = info.samplerate
                row['channels'] = info.channels
            except Exception as e:
                print(f"skip (unreadable audio): {row['audio_path']} ({e})")
                continue
            out_rows.append(row)
            n_out += 1
    _write_jsonl(output, out_rows)
    print(f'Read {n_in} rows across {len(inputs)} manifest(s). Wrote {n_out} valid rows to {output}')
    return output


def _qc_metrics(audio_path):
    import numpy as np
    import soundfile as sf
    data, sr = sf.read(audio_path, always_2d=False)
    if data.ndim > 1:
        data = data.mean(axis=1)
    duration = len(data) / sr
    peak = np.max(np.abs(data)) if len(data) else 0.0
    rms = np.sqrt(np.mean(data ** 2)) if len(data) else 0.0
    peak_db = 20 * np.log10(max(peak, 1e-9))
    rms_db = 20 * np.log10(max(rms, 1e-9))
    clipping_ratio = float(np.mean(np.abs(data) >= 0.999)) if len(data) else 1.0
    frame_len = max(int(0.02 * sr), 1)
    n_frames = len(data) // frame_len
    if n_frames == 0:
        silence_ratio = 1.0
    else:
        frames = data[:n_frames * frame_len].reshape(n_frames, frame_len)
        frame_rms = np.sqrt(np.mean(frames ** 2, axis=1))
        frame_db = 20 * np.log10(np.maximum(frame_rms, 1e-9))
        silence_ratio = float(np.mean(frame_db < -40))
    return {
        'duration': round(duration, 3), 'sample_rate': sr,
        'peak_db': round(float(peak_db), 2), 'rms_db': round(float(rms_db), 2),
        'clipping_ratio': round(clipping_ratio, 4), 'silence_ratio': round(silence_ratio, 4),
    }


def _qc_grade(metrics, text):
    dur, sr = metrics['duration'], metrics['sample_rate']
    clip, sil, rms_db = metrics['clipping_ratio'], metrics['silence_ratio'], metrics['rms_db']
    if dur < 0.5 or dur > 20 or not text or len(text.strip()) < 2 or clip > 0.01 or sr < 16000 or sil > 0.6:
        return 'D'
    if sr >= 22050 and clip == 0 and sil < 0.2 and -30 <= rms_db <= -12:
        return 'A'
    if sil < 0.4 and rms_db > -35:
        return 'B'
    return 'C'


def audio_qc(manifest, output):
    grade_counts = {'A': 0, 'B': 0, 'C': 0, 'D': 0}
    out_rows = []
    for line_no, row in enumerate(_read_jsonl(manifest), 1):
        try:
            metrics = _qc_metrics(row['audio_path'])
        except Exception as e:
            print(f'line {line_no}: failed to read audio ({e}), grading D')
            metrics = {'duration': row.get('duration', 0), 'sample_rate': 0, 'peak_db': -99,
                       'rms_db': -99, 'clipping_ratio': 1.0, 'silence_ratio': 1.0}
        row.update(metrics)
        row['quality_grade'] = _qc_grade(metrics, row.get('text', ''))
        grade_counts[row['quality_grade']] += 1
        out_rows.append(row)
        if line_no % 1000 == 0:
            print(f'  ... scored {line_no} files')
    _write_jsonl(output, out_rows)
    total = sum(grade_counts.values())
    print(f'Scored {total} files.')
    for g in 'ABCD':
        pct = 100 * grade_counts[g] / total if total else 0
        print(f'  {g}: {grade_counts[g]} ({pct:.1f}%)')
    return output


def _denoise_file(input_path, output_path):
    import shutil
    try:
        from df.enhance import enhance, init_df, load_audio, save_audio
    except ImportError:
        shutil.copyfile(input_path, output_path)
        return False
    if not hasattr(_denoise_file, '_model'):
        _denoise_file._model, _denoise_file._df_state, _ = init_df()
    audio, _ = load_audio(input_path, sr=_denoise_file._df_state.sr())
    enhanced = enhance(_denoise_file._model, _denoise_file._df_state, audio)
    save_audio(output_path, enhanced, _denoise_file._df_state.sr())
    return True


def denoise(manifest, output_dir, output_manifest, min_grade_to_keep='B'):
    import shutil
    os.makedirs(output_dir, exist_ok=True)
    grade_order = {'A': 0, 'B': 1, 'C': 2, 'D': 3}
    threshold = grade_order[min_grade_to_keep]
    n_kept, n_denoised, n_rejected, out_rows = 0, 0, 0, []
    for row in _read_jsonl(manifest):
        g = row.get('quality_grade', 'D')
        if grade_order.get(g, 3) > threshold:
            n_rejected += 1
            continue
        fname = os.path.basename(row['audio_path'])
        out_path = os.path.join(output_dir, fname)
        if g == 'A':
            shutil.copyfile(row['audio_path'], out_path)
        else:
            if _denoise_file(row['audio_path'], out_path):
                n_denoised += 1
        row['audio_path'] = out_path
        out_rows.append(row)
        n_kept += 1
    _write_jsonl(output_manifest, out_rows)
    print(f'Kept: {n_kept}  (denoised: {n_denoised})  Rejected: {n_rejected}')
    if n_denoised == 0:
        print('Note: DeepFilterNet not installed -- B-grade files were copied as-is.')
    return output_manifest


def _vad_trim_file(audio_path, out_path, min_speech_ratio):
    import shutil
    import soundfile as sf
    try:
        import torch as _torch
        _torch.set_num_threads(1)
        model, utils = _torch.hub.load(repo_or_dir='snakers4/silero-vad', model='silero_vad', trust_repo=True)
        (get_speech_timestamps, _, read_audio, *_rest) = utils
    except Exception:
        shutil.copyfile(audio_path, out_path)
        return True, 1.0
    wav = read_audio(audio_path, sampling_rate=16000)
    timestamps = get_speech_timestamps(wav, model, sampling_rate=16000)
    if not timestamps:
        return False, 0.0
    total_speech = sum(t['end'] - t['start'] for t in timestamps)
    speech_ratio = total_speech / len(wav)
    if speech_ratio < min_speech_ratio:
        return False, speech_ratio
    start, end = timestamps[0]['start'], timestamps[-1]['end']
    data, sr = sf.read(audio_path)
    ratio = sr / 16000
    orig_start, orig_end = int(start * ratio), int(end * ratio)
    trimmed_audio = data[orig_start:orig_end, :] if data.ndim > 1 else data[orig_start:orig_end]
    sf.write(out_path, trimmed_audio, sr)
    return True, speech_ratio


def vad_trim(manifest, output_dir, output_manifest, min_speech_ratio=0.4):
    import soundfile as sf
    os.makedirs(output_dir, exist_ok=True)
    n_kept, n_rejected, out_rows = 0, 0, []
    for row in _read_jsonl(manifest):
        fname = os.path.basename(row['audio_path'])
        out_path = os.path.join(output_dir, fname)
        kept, ratio = _vad_trim_file(row['audio_path'], out_path, min_speech_ratio)
        if not kept:
            n_rejected += 1
            continue
        try:
            info = sf.info(out_path)
            row['duration'] = round(info.frames / info.samplerate, 3)
        except Exception:
            pass
        row['audio_path'] = out_path
        row['speech_ratio'] = round(ratio, 4)
        out_rows.append(row)
        n_kept += 1
    _write_jsonl(output_manifest, out_rows)
    print(f'Kept: {n_kept}  Rejected (too little speech): {n_rejected}')
    return output_manifest


def _ffmpeg_loudnorm(input_path, output_path, target_lufs=-20.0, true_peak=-1.5, lra=11.0):
    import subprocess as sp
    cmd = ['ffmpeg', '-y', '-i', input_path, '-af',
           f'loudnorm=I={target_lufs}:TP={true_peak}:LRA={lra}',
           '-ac', '1', '-ar', '24000', '-sample_fmt', 's16', output_path]
    result = sp.run(cmd, stdout=sp.DEVNULL, stderr=sp.PIPE)
    return result.returncode == 0


def loudness_normalize(manifest, output_dir, output_manifest, target_lufs=-20.0):
    os.makedirs(output_dir, exist_ok=True)
    n_ok, n_fail, out_rows = 0, 0, []
    for row in _read_jsonl(manifest):
        fname = os.path.splitext(os.path.basename(row['audio_path']))[0] + '.wav'
        out_path = os.path.join(output_dir, fname)
        if not _ffmpeg_loudnorm(row['audio_path'], out_path, target_lufs):
            print(f"ffmpeg failed on {row['audio_path']}")
            n_fail += 1
            continue
        row['audio_path'] = out_path
        row['sample_rate'] = 24000
        out_rows.append(row)
        n_ok += 1
        if n_ok % 500 == 0:
            print(f'  ... normalized {n_ok} files')
    _write_jsonl(output_manifest, out_rows)
    print(f'Done. OK: {n_ok}  Failed: {n_fail}')
    return output_manifest


def normalize_text_manifest(manifest, output):
    n_ok, n_empty, out_rows = 0, 0, []
    for row in _read_jsonl(manifest):
        raw_text = row['text']
        normalized = normalize_khmer_text(raw_text)
        if not normalized.strip():
            n_empty += 1
            continue
        row['text_raw'] = raw_text
        row['text'] = normalized
        out_rows.append(row)
        n_ok += 1
    _write_jsonl(output, out_rows)
    print(f'Normalized {n_ok} transcripts. Skipped {n_empty} empty results.')
    return output


def make_splits(manifest, out_prefix, valid_frac=0.02, test_frac=0.02, seed=42):
    import random
    random.seed(seed)
    rows = _read_jsonl(manifest)
    random.shuffle(rows)
    n = len(rows)
    n_valid, n_test = int(n * valid_frac), int(n * test_frac)
    valid_rows = rows[:n_valid]
    test_rows = rows[n_valid:n_valid + n_test]
    train_rows = rows[n_valid + n_test:]
    _write_jsonl(f'{out_prefix}_train.jsonl', train_rows)
    _write_jsonl(f'{out_prefix}_valid.jsonl', valid_rows)
    _write_jsonl(f'{out_prefix}_test.jsonl', test_rows)
    total_hours = sum(r['duration'] for r in rows) / 3600
    train_hours = sum(r['duration'] for r in train_rows) / 3600
    print(f'Total: {n} utterances, {total_hours:.2f} hours')
    print(f'  Train: {len(train_rows)} ({train_hours:.2f}h)  Valid: {len(valid_rows)}  Test: {len(test_rows)}')
    return f'{out_prefix}_train.jsonl'


REQUIRED_FIELDS = ['audio_path', 'text', 'speaker_id', 'duration']


def validate_manifest(manifest_path):
    n_ok, n_bad, errors = 0, 0, []
    for line_no, row in enumerate(_read_jsonl(manifest_path), 1):
        missing = [k for k in REQUIRED_FIELDS if k not in row]
        if missing:
            errors.append(f'line {line_no}: missing fields {missing}')
            n_bad += 1
            continue
        if not row['text'] or not row['text'].strip():
            errors.append(f'line {line_no}: empty text')
            n_bad += 1
            continue
        if not isinstance(row['duration'], (int, float)) or row['duration'] <= 0:
            errors.append(f"line {line_no}: bad duration {row['duration']!r}")
            n_bad += 1
            continue
        if not os.path.exists(row['audio_path']):
            errors.append(f"line {line_no}: audio_path not found: {row['audio_path']}")
            n_bad += 1
            continue
        n_ok += 1
    print(f'Manifest: {manifest_path}\n  OK rows: {n_ok}\n  Bad rows: {n_bad}')
    if errors:
        print('  First 20 errors:')
        for e in errors[:20]:
            print('   -', e)
    if n_bad:
        raise RuntimeError(f'{n_bad} bad rows in {manifest_path} -- fix before spending GPU time on it.')


def sanitize_speaker_id(raw_id: str) -> str:
    safe = ''.join(c if c.isalnum() else '_' for c in str(raw_id))
    return safe or 'unknown_speaker'


def convert_to_fish_format(manifest, out_dir, copy=True):
    import shutil
    from collections import defaultdict
    os.makedirs(out_dir, exist_ok=True)
    speaker_counters = defaultdict(int)
    n_written = 0
    for row in _read_jsonl(manifest):
        speaker = sanitize_speaker_id(row['speaker_id'])
        speaker_dir = os.path.join(out_dir, speaker)
        os.makedirs(speaker_dir, exist_ok=True)
        speaker_counters[speaker] += 1
        stem = f'{speaker_counters[speaker]:08d}'
        wav_out = os.path.join(speaker_dir, f'{stem}.wav')
        lab_out = os.path.join(speaker_dir, f'{stem}.lab')
        src_wav = os.path.abspath(row['audio_path'])
        if copy:
            shutil.copyfile(src_wav, wav_out)
        else:
            if os.path.lexists(wav_out):
                os.remove(wav_out)
            os.symlink(src_wav, wav_out)
        with open(lab_out, 'w', encoding='utf-8') as lab_f:
            lab_f.write(row['text'].strip() + '\n')
        n_written += 1
    print(f'Wrote {n_written} wav/lab pairs across {len(speaker_counters)} speakers.')
    print(f'Speakers: {dict(speaker_counters)}')
    return out_dir


print('Data pipeline functions defined.')

## 8 · Progressive multi-round training: helper to download + rebuild the dataset each round

In [ ]:
import shutil


def _report_disk_usage(label=""):
    total, used, free = shutil.disk_usage(WORKDIR)
    gb = 1024 ** 3
    print(f"[disk{' -- ' + label if label else ''}] "
          f"used {used / gb:.1f}GB / {total / gb:.1f}GB  (free: {free / gb:.1f}GB)")


def _cleanup_dir(path):
    """Delete a pipeline stage's audio directory once the next stage has
    already consumed it. Each stage writes a FULL fresh copy of the audio
    rather than modifying in place, so without this, raw -> denoised ->
    VAD-trimmed -> loudness-normalized -> Fish-format ends up as 5 near-full
    copies of the same growing dataset sitting on disk at once. Safe to
    delete: every later manifest stores absolute paths into its OWN output
    dir, never back into an earlier stage's dir, so nothing still points here."""
    if os.path.isdir(path):
        shutil.rmtree(path, ignore_errors=True)
        print(f'  (freed {path})')


def run_pipeline_round(cumulative_samples):
    """Download up to `cumulative_samples` DDD clips (cumulative) and rebuild
    the Fish-format speaker-folder dataset from the full accumulated set.
    Deletes each stage's audio directory as soon as the next stage is done
    with it, so only one stage's worth of audio exists on disk at a time.
    Returns the Fish-format dataset dir."""
    raw_dir = os.path.join(WORKDIR, 'data', 'raw', 'ddd')
    denoised_dir = os.path.join(WORKDIR, 'data', 'processed', 'ddd_denoised')
    vad_dir = os.path.join(WORKDIR, 'data', 'processed', 'ddd_vad')
    clean_dir = os.path.join(WORKDIR, 'data', 'processed', 'ddd_24k')

    m_raw = download_ddd(DATASET, 'train', raw_dir,
                          max_samples=cumulative_samples, max_shard_files=MAX_SHARD_FILES)
    m_merged = export_audio([m_raw], os.path.join(MANIFEST_DIR, 'ddd_raw_merged.jsonl'))
    m_qc = audio_qc(m_merged, os.path.join(MANIFEST_DIR, 'ddd_qc.jsonl'))

    m_denoised = denoise(m_qc, denoised_dir, os.path.join(MANIFEST_DIR, 'ddd_denoised.jsonl'))
    _cleanup_dir(raw_dir)  # denoise() already read every raw file it needed

    m_vad = vad_trim(m_denoised, vad_dir, os.path.join(MANIFEST_DIR, 'ddd_vad.jsonl'))
    _cleanup_dir(denoised_dir)

    m_clean = loudness_normalize(m_vad, clean_dir, os.path.join(MANIFEST_DIR, 'ddd_clean.jsonl'))
    _cleanup_dir(vad_dir)

    m_normalized = normalize_text_manifest(m_clean, os.path.join(MANIFEST_DIR, 'ddd_normalized.jsonl'))
    m_train = make_splits(m_normalized, os.path.join(MANIFEST_DIR, 'ddd'))
    validate_manifest(m_train)

    fish_dataset_dir = os.path.join(WORKDIR, 'data', 'fish', 'khmer_base')
    convert_to_fish_format(m_train, fish_dataset_dir, copy=True)
    _cleanup_dir(clean_dir)

    _report_disk_usage('after pipeline round')
    return fish_dataset_dir


PROTO_DIR = os.path.join(WORKDIR, 'data', 'fish', 'khmer_base_protos')
print('run_pipeline_round() defined.')

## 9 · Run all training rounds

Runs all of `STAGE1_ROUNDS` (see CONFIG cell) back to back. This trains **one**
model across all speakers in the accumulated `data/fish/khmer_base/` (not one
model per person -- see the intro markdown).

In [ ]:
FISH_DATASET_DIR = os.path.join(WORKDIR, 'data', 'fish', 'khmer_base')

if START_ROUND <= 1:
    CUR_CKPT = CHECKPOINT_DIR  # round 1 starts from the raw pretrained Fish Speech checkpoint
else:
    CUR_CKPT = os.path.join(WORKDIR, 'models', 'khmer_base', f'round{START_ROUND - 1}', 'merged')
    if not os.path.isdir(CUR_CKPT):
        raise RuntimeError(
            f'START_ROUND={START_ROUND} but {CUR_CKPT} does not exist -- '
            f'round {START_ROUND - 1} must finish (and merge) before resuming from round {START_ROUND}.'
        )
    print(f'Resuming from round {START_ROUND}, starting checkpoint: {CUR_CKPT}')

for round_idx, round_cfg in enumerate(STAGE1_ROUNDS, start=1):
    if round_idx < START_ROUND:
        print(f'Skipping round {round_idx} (already done -- START_ROUND={START_ROUND})')
        continue

    n_samples, n_steps = round_cfg['cumulative_samples'], round_cfg['steps']
    print(f"\n===== Round {round_idx}/{len(STAGE1_ROUNDS)}: "
          f"up to {n_samples} cumulative samples, {n_steps} steps, "
          f"starting from {CUR_CKPT} =====")

    FISH_DATASET_DIR = run_pipeline_round(n_samples)

    print('-- VQ token extraction (skips files already extracted) --')
    # --num-workers must stay 1: extract_vq.py's own multi-worker mode spawns
    # separate subprocesses and .wait()s for them WITHOUT checking their exit
    # codes (see its main(), the `for p in processes: p.wait()` loop), so any
    # worker crash (e.g. from N processes contending for one GPU) is silently
    # swallowed and looks like success -- exactly what produced zero .npy
    # files here. num_workers=1 skips that spawning path entirely.
    run_step_streaming([sys.executable, os.path.join(FISH_DIR, 'tools', 'vqgan', 'extract_vq.py'),
              FISH_DATASET_DIR, '--num-workers', '1', '--batch-size', '4',
              '--config-name', 'modded_dac_vq',
              '--checkpoint-path', os.path.join(CHECKPOINT_DIR, 'codec.pth')],
              label='VQ token extraction')

    n_npy = sum(1 for _ in __import__('pathlib').Path(FISH_DATASET_DIR).rglob('*.npy'))
    if n_npy == 0:
        raise RuntimeError(
            f'VQ extraction produced 0 .npy files under {FISH_DATASET_DIR} -- '
            'training would fail later with an opaque ZeroDivisionError, so '
            'failing now instead. Check the extraction log above for per-file errors.'
        )
    print(f'{n_npy} .npy files present after VQ extraction.')

    print('-- Build protobuf dataset (full accumulated set) --')
    run_step_streaming([sys.executable, os.path.join(FISH_DIR, 'tools', 'llama', 'build_dataset.py'),
              '--input', FISH_DATASET_DIR, '--output', PROTO_DIR,
              '--text-extension', '.lab', '--num-workers', '4'],
              label='Build protobuf dataset')

    n_shards = len(list(__import__('pathlib').Path(PROTO_DIR).glob('*.protos')))
    if n_shards == 0:
        raise RuntimeError(
            f'Build protobuf dataset produced 0 shard files under {PROTO_DIR} -- '
            'would fail later with an opaque ZeroDivisionError in training, so '
            'failing now instead. Check the build log above.'
        )
    print(f'{n_shards} .protos shard file(s) present.')

    project_name = f'khmer_base_round{round_idx}'
    round_dir = os.path.join(WORKDIR, 'models', 'khmer_base', f'round{round_idx}')
    os.makedirs(round_dir, exist_ok=True)

    print(f'-- LoRA fine-tune: {n_steps} steps from {CUR_CKPT} --')
    _train_env = os.environ.copy()
    # Reduces allocator fragmentation (suggested directly by PyTorch's own OOM
    # message) -- cheap to set, doesn't hurt if it wasn't the actual issue.
    _train_env['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
    run_step_streaming(
        [sys.executable, os.path.join(FISH_DIR, 'fish_speech', 'train.py'),
         '--config-name', 'text2semantic_finetune',
         f'project={project_name}',
         f'+lora@model.model.lora_config={LORA_PRESET}',
         f'train_dataset.proto_files=[{PROTO_DIR}]',
         f'val_dataset.proto_files=[{PROTO_DIR}]',
         f'pretrained_ckpt_path={CUR_CKPT}',
         f'trainer.max_steps={n_steps}',
         'trainer.val_check_interval=100',
         # Defaults (batch_size=4, max_length=4096) OOM the 862M-param backbone
         # on a 14.56GB T4 even with only LoRA adapters trainable -- the full
         # frozen model still runs forward/backward, so activation memory
         # scales with batch_size * sequence_length. Cut both.
         'data.batch_size=1',
         'max_length=1024',
         f'hydra.run.dir={round_dir}'],
        cwd=FISH_DIR,
        env=_train_env,
        label=f'Round {round_idx} training',
    )

    print('-- Merge this round\'s LoRA into the checkpoint the next round starts from --')
    ckpt_dir = os.path.join(FISH_DIR, 'results', project_name, 'checkpoints')
    ckpts = sorted(
        (os.path.join(ckpt_dir, f) for f in os.listdir(ckpt_dir)) if os.path.isdir(ckpt_dir) else [],
        key=os.path.getmtime,
    )
    if not ckpts:
        raise RuntimeError(f'No LoRA checkpoint found under {ckpt_dir}')
    latest_ckpt = ckpts[-1]
    print('Merging', latest_ckpt)
    merged_out = os.path.join(round_dir, 'merged')
    run_step([sys.executable, os.path.join(FISH_DIR, 'tools', 'llama', 'merge_lora.py'),
              '--lora-config', LORA_PRESET, '--base-weight', CUR_CKPT,
              '--lora-weight', latest_ckpt, '--output', merged_out])

    CUR_CKPT = merged_out
    print(f'Round {round_idx} done. Checkpoint at {CUR_CKPT}')

MERGED_CKPT = CUR_CKPT
print('\nAll rounds complete. Final checkpoint:', MERGED_CKPT)

### Optional: back up the final checkpoint (Drive / Kaggle Output)

On Colab, `WORKDIR` is local ephemeral disk (see section 2), so `MERGED_CKPT` is
wiped if the runtime disconnects. On Kaggle, `WORKDIR` is `/kaggle/temp`, which is
scratch space wiped when the session ends -- only files under `/kaggle/working` are
kept as the notebook's persisted Output (and are what you can download afterwards).
Run this cell to copy just the final trained checkpoint (not the whole
dataset/engine) somewhere that survives -- it's much smaller than the full working
directory.

In [ ]:
BACKUP_CHECKPOINT = False  # set True to copy MERGED_CKPT somewhere persistent

if BACKUP_CHECKPOINT:
    import shutil
    if IN_COLAB:
        from google.colab import drive
        drive.mount('/content/drive')
        dest = '/content/drive/MyDrive/khmer-voice-clone-checkpoints/khmer_base_merged'
    elif IN_KAGGLE:
        # /kaggle/working is the only directory whose contents persist as this
        # notebook's Output after the session ends / is committed.
        dest = '/kaggle/working/khmer_base_merged'
    else:
        dest = None
        print('BACKUP_CHECKPOINT is only meaningful on Colab/Kaggle (WORKDIR is already persistent elsewhere).')

    if dest:
        os.makedirs(os.path.dirname(dest), exist_ok=True)
        if os.path.isdir(dest):
            shutil.rmtree(dest)
        shutil.copytree(MERGED_CKPT, dest)
        print('Backed up to', dest)

## 10 · List available speakers, then fill in `PERSON_MAP` above

Each subfolder of `data/fish/khmer_base/` is one DDD speaker, with numbered
`.wav`/`.lab` pairs (this reflects the FULL accumulated dataset after all rounds).
Pick two (or more) speaker folder names, put them in `PERSON_MAP` in the CONFIG
cell, then re-run from here down (no need to retrain -- generation just needs a
reference clip).

In [ ]:
speakers = sorted(os.listdir(FISH_DATASET_DIR))
print(f'{len(speakers)} speakers available:')
for sp in speakers:
    n_clips = len([f for f in os.listdir(os.path.join(FISH_DATASET_DIR, sp)) if f.endswith('.wav')])
    print(f'  {sp}  ({n_clips} clips)')

missing = [name for name, sid in PERSON_MAP.items() if not sid]
if missing:
    print(f'\n>>> Fill in PERSON_MAP for: {missing} (using one of the speaker ids above), then re-run.')

## 11 · Load the model once for generation

In [ ]:
import sys
sys.path.insert(0, FISH_DIR)

from tools.server.model_manager import ModelManager
from fish_speech.utils.schema import ServeTTSRequest, ServeReferenceAudio

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

manager = ModelManager(
    mode='tts',
    device=DEVICE,
    half=False,
    compile=False,
    llama_checkpoint_path=MERGED_CKPT,
    decoder_checkpoint_path=os.path.join(CHECKPOINT_DIR, 'codec.pth'),
    decoder_config_name='modded_dac_vq',
)
print('Model loaded on', DEVICE)

## 12 · Generate: pick a person by name, give it text

In [ ]:
import numpy as np
import soundfile as sf


def _pick_reference_clip(speaker_id: str):
    """Use the first available wav/lab pair for this speaker as the
    in-context voice reference."""
    speaker_dir = os.path.join(FISH_DATASET_DIR, speaker_id)
    if not os.path.isdir(speaker_dir):
        raise ValueError(f"No speaker folder {speaker_dir} -- check PERSON_MAP against section 9's list.")
    stems = sorted(f[:-4] for f in os.listdir(speaker_dir) if f.endswith('.wav'))
    if not stems:
        raise ValueError(f'No wav files under {speaker_dir}')
    stem = stems[0]
    with open(os.path.join(speaker_dir, f'{stem}.lab'), encoding='utf-8') as f:
        ref_text = f.read().strip()
    with open(os.path.join(speaker_dir, f'{stem}.wav'), 'rb') as f:
        ref_audio_bytes = f.read()
    return ref_audio_bytes, ref_text


def speak(text: str, speaker: str, out_path: str = None):
    """Generate `text` in `speaker`'s voice (a PERSON_MAP key, e.g. "person1")."""
    if speaker not in PERSON_MAP or not PERSON_MAP[speaker]:
        raise ValueError(f'{speaker!r} not set in PERSON_MAP -- see section 9.')
    speaker_id = PERSON_MAP[speaker]
    ref_audio_bytes, ref_text = _pick_reference_clip(speaker_id)

    normalized_text = normalize_khmer_text(text)
    req = ServeTTSRequest(
        text=normalized_text,
        references=[ServeReferenceAudio(audio=ref_audio_bytes, text=ref_text)],
    )

    audio_chunks, sample_rate = [], None
    for result in manager.tts_inference_engine.inference(req):
        if result.code == 'error':
            raise result.error
        if result.code in ('segment', 'final') and result.audio is not None:
            sample_rate, chunk = result.audio
            audio_chunks.append(chunk)
    if not audio_chunks:
        raise RuntimeError('No audio produced.')

    full_audio = np.concatenate(audio_chunks)
    if out_path is None:
        out_path = os.path.join(WORKDIR, 'outputs', 'eval', f'{speaker}.wav')
    os.makedirs(os.path.dirname(out_path) or '.', exist_ok=True)
    sf.write(out_path, full_audio, sample_rate)
    print(f'{speaker} ({speaker_id}) -> {out_path}  ({len(full_audio) / sample_rate:.2f}s)')
    return out_path

In [ ]:
from IPython.display import Audio, display

TEXT = "សួស្តី​ពិភពលោក"  # <- put your own Khmer text here

for person in PERSON_MAP:
    path = speak(TEXT, speaker=person)
    display(Audio(path))